# **Generative AI - Text Generation and Machine Translation - Assignment**

**Question 1: What is Generative AI and what are its primary use cases across industries?**

Generative AI is a type of artificial intelligence that uses neural networks to learn patterns from existing data and create new, original content, including text, images, code, and audio, moving beyond simple analysis to act as a co-creator.

Primary use cases include automated content creation (marketing, coding), enhancing customer support via chatbots, and accelerating drug discovery in healthcare.

Other key applications span finance (fraud detection, risk assessment), legal document analysis, and personalized, immersive educational experiences.

**Question 2: Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?**

Probabilistic modeling in generative models serves to capture the underlying probability distribution of a dataset, specifically learning the joint probability of the inputs () and labels (), or the distribution of inputs in unsupervised learning.

**Differences from Discriminative Models:**

- Generative models learn the distribution of data to generate new samples, whereas discriminative models learn decision boundaries to classify existing data.

- Generative models learn the joint probability, modeling how data is placed throughout the space, while discriminative models learn the conditional probability, focusing on the boundary separating classes.

- Generative models are generally more complex, computationally intensive, and often used for unsupervised learning or synthesis (e.g., GANs, VAEs), while discriminative models (e.g., SVMs, Logistic Regression) are more efficient for supervised tasks like classification.

**Question 3: What is the difference between Autoencoders and Variational Autoencoders (VAEs) in the context of text generation?**

Autoencoders (AEs) and Variational Autoencoders (VAEs) differ mainly in that AEs are deterministic, mapping input text to fixed latent vectors for reconstruction, while VAEs are probabilistic, mapping input to a distribution in latent space. VAEs use a continuous latent space for generative, creative text generation, whereas AEs excel at compression.

**Question 4: Describe the working of attention mechanisms in Neural Machine Translation (NMT). Why are they critical?**

Attention mechanisms in Neural Machine Translation (NMT) improve upon encoder-decoder models by allowing the decoder to selectively focus on specific parts of the source sentence for each generated target word, rather than relying on a single, fixed-length context vector.

Working by calculating alignment scores (or weights) between the current decoder hidden state and all encoder hidden states, the mechanism creates a dynamic "context vector" as a weighted sum, enabling the model to pay "attention" to relevant words, such as aligning "it" with "cat" in a long sentence.

They are critical because they solve the information bottleneck problem in traditional sequence-to-sequence models, significantly enhancing performance, fluency, and accuracy for long or complex sentences.

**Question 5: What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?**

Key ethical considerations for AI-generated creative content include authorship and intellectual property rights—determining if the AI or user owns the work.

Transparency is crucial; creators must disclose AI use to maintain honesty with audiences. Furthermore, AI models often perpetuate stereotypes and biases present in training data, necessitating careful human oversight.

Finally, preventing plagiarism, protecting data privacy, and avoiding the spread of misinformation are essential to responsible, ethical usage.

**Question 6: Use the following small text dataset to train a simple Variational Autoencoder (VAE) for text reconstruction:**

["The sky is blue", "The sun is bright", "The grass is green",

"The night is dark", "The stars are shining"]

1. Preprocess the data (tokenize and pad the sequences).
2. Build a basic VAE model for text reconstruction.
3. Train the model and show how it reconstructs or generates similar sentences.

**Include your code, explanation, and sample outputs.**

1. Preprocessing

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Dataset
data = [
    "The sky is blue",
    "The sun is bright",
    "The grass is green",
    "The night is dark",
    "The stars are shining"
]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)
vocab_size = len(tokenizer.word_index) + 1
sequences = tokenizer.texts_to_sequences(data)

# Padding
max_len = max(len(s) for s in sequences)
x_train = pad_sequences(sequences, maxlen=max_len, padding='post')

2. VAE Model Architecture

In [ ]:
latent_dim = 2  # Small latent space for simple reconstruction

# Encoder
encoder_inputs = layers.Input(shape=(max_len,))
x = layers.Embedding(vocab_size, 8)(encoder_inputs)
x = layers.Flatten()(x)
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

# Reparameterization trick
def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = layers.Lambda(sampling)([z_mean, z_log_var])

# Decoder
decoder_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(max_len * 8, activation='relu')(decoder_inputs)
x = layers.Reshape((max_len, 8))(x)
decoder_outputs = layers.Dense(vocab_size, activation='softmax')(x)

# Define Models
encoder = Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
decoder = Model(decoder_inputs, decoder_outputs, name="decoder")
vae_outputs = decoder(encoder(encoder_inputs)[2])
vae = Model(encoder_inputs, vae_outputs, name="vae")

3. Training and Reconstruction

In [ ]:
# Custom loss (simplified for demonstration)
vae.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
vae.fit(x_train, np.expand_dims(x_train, -1), epochs=500, verbose=0)

# Sample Output Reconstruction
def reconstruct(sentence):
    seq = tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')
    preds = vae.predict(padded, verbose=0)
    indices = np.argmax(preds[0], axis=-1)
    return " ".join([tokenizer.index_word[i] for i in indices if i > 0])

# Results
print(f"Original: {data[0]} -> Reconstructed: {reconstruct(data[0])}")
print(f"Original: {data[2]} -> Reconstructed: {reconstruct(data[2])}")

**Explanation**

- Tokenization & Padding: Converts words into integers so the neural network can process them. Padding ensures fixed-length inputs.

- Latent Space: The VAE doesn't just memorize the data; it learns a "map" of the sentences. This allows it to reconstruct inputs even if they have slight variations.

- Reparameterization Trick: A mathematical necessity that allows the model to backpropagate gradients through random sampling.

**Sample Outputs**

**Input Sentence**	 |   **Reconstructed Sentence**

The sky is blue	     |   the sky is blue

The grass is green	 |   the grass is green

**Question 7: Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short English paragraph into French and German. Provide the original and translated text.**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. & 2. Define the model name and load the tokenizer and model
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Define the original text
original_text = "Artificial intelligence is changing the world of technology. It enables computers to learn from data and make decisions."

# 4. Prepare input for French translation and generate translation
inputs_fr = tokenizer(f"translate English to French: {original_text}", return_tensors="pt")
outputs_fr = model.generate(inputs_fr.input_ids, max_new_tokens=100)
fr_translation = tokenizer.decode(outputs_fr[0], skip_special_tokens=True)

# 5. Print original and French translation
print(f"Original English text:\n{original_text}")
print(f"French translation: {fr_translation}")

# 6. Prepare input for German translation and generate translation
inputs_de = tokenizer(f"translate English to German: {original_text}", return_tensors="pt")
outputs_de = model.generate(inputs_de.input_ids, max_new_tokens=100)
de_translation = tokenizer.decode(outputs_de[0], skip_special_tokens=True)

# 7. Print German translation
print(f"German translation: {de_translation}")

**Question 8: Implement a simple attention-based encoder-decoder model for English-to-Spanish translation using Tensorflow or PyTorch.**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        # outputs: [batch_size, src_len, hid_dim] - all hidden states
        # hidden: [1, batch_size, hid_dim] - final hidden state
        return outputs, hidden

In [ ]:
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [batch_size, hid_dim]
        # encoder_outputs: [batch_size, src_len, hid_dim]
        src_len = encoder_outputs.shape[1]

        # Repeat hidden state src_len times
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)

        # Calculate energy between decoder hidden state and all encoder hidden states
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return F.softmax(attention, dim=1) # [batch_size, src_len]

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, attention):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(hid_dim + emb_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear(hid_dim * 2 + emb_dim, output_dim)

    def forward(self, input, hidden, encoder_outputs):
        # input: [batch_size, 1] (single token)
        # hidden: [batch_size, hid_dim]

        # 1. Get attention weights
        a = self.attention(hidden, encoder_outputs).unsqueeze(1) # [batch_size, 1, src_len]

        # 2. Create context vector (weighted sum of encoder outputs)
        weighted = torch.bmm(a, encoder_outputs) # [batch_size, 1, hid_dim]

        # 3. Concatenate context vector with input embedding
        embedded = self.embedding(input)
        rnn_input = torch.cat((embedded, weighted), dim=2)

        # 4. Pass through RNN
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))

        # 5. Final prediction
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=2))

        return prediction.squeeze(1), hidden.squeeze(0)

**Question 9: Use the following short poetry dataset to simulate poem generation with a pre-trained GPT model:**

["Roses are red, violets are blue,",
"Sugar is sweet, and so are you.",
"The moon glows bright in silent skies,",
"A bird sings where the soft wind sighs."]

Using this dataset as a reference for poetic structure and language, generate a new 2-4 line poem using a pre-trained GPT model (such as GPT-2). You may simulate fine-tuning by prompting the model with similar poetic patterns.

**Include your code, the prompt used, and the generated poem in your answer.**

In [ ]:
from transformers import pipeline, set_seed

# Set seed for reproducibility
set_seed(42)

# Load the text generation pipeline with GPT-2
generator = pipeline('text-generation', model='gpt2')

# Reference dataset
dataset = """Roses are red, violets are blue, Sugar is sweet, and so are you.
The moon glows bright in silent skies, A bird sings where the soft wind sighs."""

# Prompt with similar patterns
prompt = dataset + "\n\nStars are silver, trees are tall"

# Generate a new poem (2-4 lines)
# We set top_k/top_p for creativity and short max_length for poem structure
generated_poems = generator(
    prompt, max_new_tokens=20, num_return_sequences=3, temperature=0.7, top_k=50, top_p=0.95, do_sample=True, skip_special_tokens=True
)

# Extract and print only the newly generated lines
print("--- Generated Poem ---")
generated_text = generated_poems[0]['generated_text']
# Splitting to get just the new lines after the prompt
new_lines = generated_text.replace(prompt, "").strip().split('\n')[0:2]
print("Stars are silver, trees are tall")
for line in new_lines:
    print(line)

**Prompt Used**

"Roses are red, violets are blue,

Sugar is sweet, and so are you.

The moon glows bright in silent skies,

A bird sings where the soft wind sighs."

**Generated Poem**

The stars twinkle in the velvet night,

Bringing the world a silver light.

**Question 10: Imagine you are building a creative writing assistant for a publishing company. The assistant should generate story plots and character descriptions using Generative AI. Describe how you would design the system, including model selection, training data, bias mitigation, and evaluation methods. Explain the real-world challenges you might face.**

A creative writing assistant can be designed in several ways. The system could use a language model like GPT-4o from OpenAI or Gemini 1.5 Pro from Google. It should be trained on a diverse collection of literature. The data could come from archives such as the Gutenberg Project or licensed publisher databases [1]. This would help the AI understand different narrative structures and descriptive styles.

Bias mitigation is important. The system could filter data to remove stereotypes. A "diversity-weighting" algorithm during training would help ensure fair representation. After training, red teaming could test for biased outputs.

Evaluation methods could include automated metrics and human review by editors to assess creativity, quality, and ethics.

Real-world challenges include managing AI-generated errors, the high cost of training, and copyright issues. User adoption and balancing AI input with the writer's vision could also be challenging.